# GeneFlow AI · Búsqueda de hiperparámetros en Kaggle

Este notebook ejecuta de principio a fin, **sin supervisión**, la búsqueda definitiva de hiperparámetros de las dos arquitecturas, aprovechando una sesión completa de Kaggle de 12 horas con **dos GPU T4**.

## Cómo lanzarlo

1. En Kaggle: **Create → New Notebook → File → Import Notebook** y sube este archivo.
2. En el panel derecho, **Session options**:
   - **Accelerator: GPU T4 ×2**
   - **Internet: On** (hace falta verificar la cuenta con un teléfono)
3. **Save Version → Save & Run All (Commit)**. Se ejecuta en segundo plano: puedes cerrar el navegador.
4. Cuando termine, abre la versión guardada y descarga la carpeta **`search`** de la pestaña **Output**.

## Qué hace

| Paso | Qué ocurre | Tiempo aproximado |
|---|---|---|
| 1 | Detecta las GPU y el driver, e instala PyTorch con la versión de CUDA compatible (13.0 o 12.6) | 3-5 min |
| 2 | Clona el repositorio, instala Python 3.14 con `uv` y las dependencias | 2-3 min |
| 3 | `geneflow prepare`: descarga las 4 fuentes, construye el dataset y genera los datos sintéticos | 10-15 min |
| 4 | `geneflow cache`: espacio de etiquetas y secuencias codificadas | 2-4 min |
| 5 | **Búsqueda: la CNN en la GPU 0 y el Transformer en la GPU 1, en paralelo** | El resto de la sesión |
| 6 | Resumen con las mejores configuraciones | 1 min |

**Presupuesto.** Cada búsqueda recibe como límite de tiempo todo lo que queda de sesión menos 30 minutos de margen. La hora límite se comprueba en cada lote, así que ninguna prueba se pasa: la que esté en curso se corta en segundos y el notebook termina a tiempo de guardar las salidas. Si Kaggle cortara por tiempo, se perderían.

**Profundidad.** Cada prueba entrena hasta 6 épocas de 100 mil secuencias. El recortador **Hyperband** no descarta nada antes de la época 2, para no castigar a los modelos grandes, que arrancan más despacio. En la época 2 compara las pruebas entre sí y corta las que van peor; las demás llegan a la época 6. Así las configuraciones malas cuestan poco y las prometedoras terminan el entrenamiento completo. La validación usa un subconjunto fijo de 20 mil secuencias de `val`, el mismo para todas las pruebas. `test` no se toca.

**Si solo hay una GPU,** las dos arquitecturas se ejecutan una detrás de otra con la mitad del tiempo cada una.

**Reanudar en otra sesión.** Añade como *Input* la salida de una ejecución anterior (**Add Input → Your Work → la versión anterior**). El paso de reanudación copia sus estudios y la búsqueda continúa donde se quedó, con valores nuevos.

## Qué se guarda en `search/`

| Archivo | Contenido |
|---|---|
| `cnn.log`, `transformer.log` | Los estudios completos de Optuna. En local, copiándolos a `checkpoints/search/`, la celda de búsqueda del notebook principal los lee y los muestra |
| `cnn-best.json`, `transformer-best.json` | La mejor configuración de cada arquitectura |
| `cnn-trials.json`, `transformer-trials.json` | Todas las pruebas con sus hiperparámetros, estado y objetivo |
| `cnn-run.log`, `transformer-run.log` | El registro completo de cada búsqueda, época a época |


In [ ]:
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

SESSION_START = time.time()
SESSION_HOURS = 12.0
SAFETY_MARGIN_HOURS = 0.5

REPOSITORY = "https://github.com/Dexaroz/geneflow-ai-taxonomy-classifier.git"
BRANCH = "main"

WORK = Path("/tmp/geneflow")
REPO = WORK / "repo"
DATA = WORK / "data"
OUTPUT = Path("/kaggle/working/search")
GENEFLOW = REPO / ".venv" / "bin" / "geneflow"
PYTHON = REPO / ".venv" / "bin" / "python"

ARCHITECTURES = ["CNN", "Transformer"]

SEARCH_ARGUMENTS = [
    "--epochs", "6",
    "--samples-per-epoch", "100000",
    "--validation-samples", "20000",
    "--batch-size", "128",
    "--precision", "fp16",
    "--gpu-memory-fraction", "0.92",
]

REPORT_EVERY_SECONDS = 300

WORK.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)


def run(command, **kwargs):
    print("$", " ".join(str(part) for part in command), flush=True)

    return subprocess.run([str(part) for part in command], check=True, **kwargs)


def elapsed_hours():
    return (time.time() - SESSION_START) / 3600


def remaining_hours():
    return SESSION_HOURS - SAFETY_MARGIN_HOURS - elapsed_hours()


print(f"Python del notebook: {sys.version.split()[0]} | trabajo en {WORK} | salidas en {OUTPUT}")

## 1-2. Entorno: GPU, PyTorch, repositorio y dependencias

In [ ]:
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=True,
)
gpus = [line.split(", ") for line in gpu_query.stdout.strip().splitlines()]
driver_major = int(gpus[0][1].split(".")[0])
cuda_index = "cu130" if driver_major >= 580 else "cu126"

for index, (name, driver, memory) in enumerate(gpus):
    print(f"GPU {index}: {name} | driver {driver} | {memory}")

print(f"PyTorch se instalará con {cuda_index}")

run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])

if REPO.exists():
    shutil.rmtree(REPO)

run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%h %s"])

run(["uv", "python", "install", "3.14"])
run(["uv", "sync", "--frozen", "--no-default-groups", "--no-install-package", "torch"], cwd=REPO)
run(["uv", "pip", "install", "--python", PYTHON, "torch==2.14.0", "--index-url", f"https://download.pytorch.org/whl/{cuda_index}"], cwd=REPO)

run([PYTHON, "-c", "import torch; print('torch', torch.__version__, '| GPU visibles:', torch.cuda.device_count(), '|', torch.cuda.get_device_name(0))"])

print(f"Entorno listo en {elapsed_hours() * 60:.0f} min")

## 3-4. Datos: descarga, construcción, datos sintéticos y cachés

In [ ]:
run([GENEFLOW, "prepare", "--data-dir", DATA])
run([GENEFLOW, "cache", "--data-dir", DATA])

build_report = json.loads((DATA / "processed" / "geneflow" / "build_report.json").read_text())
augment_report = json.loads((DATA / "processed" / "geneflow" / "augment_report.json").read_text())

print(f"Dataset: {build_report['merge']['kept']:,} secuencias | particiones: {build_report['splits']}")
print(f"Sintéticas: {augment_report['synthetic']:,} ({100 * augment_report['synthetic_ratio']:.1f} % de train)")
print(f"Datos listos a los {elapsed_hours() * 60:.0f} min de sesión")

## Reanudación (solo si se ha añadido una ejecución anterior como *Input*)

In [ ]:
previous = sorted(Path("/kaggle/input").glob("**/search/*.log"))
studies = [path for path in previous if not path.name.endswith("-run.log")]

for path in studies:
    target = OUTPUT / path.name

    if not target.exists():
        shutil.copy(path, target)
        print(f"Reanudando desde {path}")

if not studies:
    print("Sin ejecuciones anteriores: la búsqueda empieza de cero")

## 5. Búsqueda en paralelo, con un informe cada 5 minutos

In [ ]:
gpu_count = len(gpus)
parallel = gpu_count >= len(ARCHITECTURES)
groups = [ARCHITECTURES] if parallel else [[architecture] for architecture in ARCHITECTURES]


def launch(architecture, gpu, hours):
    environment = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    log = open(OUTPUT / f"{architecture.lower()}-run.log", "a")

    command = [
        GENEFLOW, "search",
        "--data-dir", DATA,
        "--architecture", architecture,
        "--storage", OUTPUT / f"{architecture.lower()}.log",
        "--hours", f"{hours:.3f}",
        *SEARCH_ARGUMENTS,
    ]

    print(f"Lanzando {architecture} en la GPU {gpu} con {hours:.2f} h de presupuesto")

    return subprocess.Popen([str(part) for part in command], stdout=log, stderr=subprocess.STDOUT, env=environment)


def trials_summary(architecture):
    path = OUTPUT / f"{architecture.lower()}-trials.json"

    if not path.exists():
        return "sin pruebas terminadas"

    rows = json.loads(path.read_text())
    states = {}

    for row in rows:
        states[row["state"]] = states.get(row["state"], 0) + 1

    values = [row["value"] for row in rows if row["value"] is not None]
    best = f"mejor {100 * max(values):.1f} %" if values else "sin completas"

    return f"{len(rows)} pruebas {states} | {best}"


def last_line(architecture):
    path = OUTPUT / f"{architecture.lower()}-run.log"
    lines = path.read_text(errors="replace").strip().splitlines() if path.exists() else []

    return lines[-1][-160:] if lines else "(sin salida todavía)"


for group in groups:
    hours = remaining_hours() / (len(groups) - groups.index(group))
    processes = {architecture: launch(architecture, gpu, hours) for gpu, architecture in enumerate(group)}

    while any(process.poll() is None for process in processes.values()):
        time.sleep(REPORT_EVERY_SECONDS)

        print(f"\n[{elapsed_hours():.2f} h de sesión, quedan {max(0.0, remaining_hours()):.2f} h de búsqueda]")

        for architecture in group:
            print(f"  {architecture}: {trials_summary(architecture)}")
            print(f"    {last_line(architecture)}")

    for architecture, process in processes.items():
        print(f"{architecture} terminó con código {process.returncode}")

print(f"\nBúsqueda terminada a las {elapsed_hours():.2f} h de sesión")

## 6. Resumen

In [ ]:
for architecture in ARCHITECTURES:
    print(f"\n=== {architecture}: {trials_summary(architecture)}")

    trials_path = OUTPUT / f"{architecture.lower()}-trials.json"
    best_path = OUTPUT / f"{architecture.lower()}-best.json"

    if trials_path.exists():
        rows = [row for row in json.loads(trials_path.read_text()) if row["value"] is not None]

        for row in sorted(rows, key=lambda item: -item["value"])[:10]:
            parameters = {key: value for key, value in row.items() if key not in {"trial", "state", "value", "epochs"}}
            print(f"  prueba {row['trial']:>3}: {100 * row['value']:5.2f} % | {parameters}")

    if best_path.exists():
        best = json.loads(best_path.read_text())
        print(f"  MEJOR: prueba {best['trial']} con {100 * best['value']:.2f} %")
        print(f"  encoder: {best['encoder']}")

print("\nArchivos para descargar (pestaña Output):")

for path in sorted(OUTPUT.iterdir()):
    print(f"  search/{path.name} ({path.stat().st_size / 1e6:.2f} MB)")